# Лёгкие цепи мыши — удаление адаптеров cutadapt и фильтрация fastp Q30/min250

`cutadapt` удаляет только явно заданные адаптеры, после чего `fastp` фильтрует целые пары с параметрами `-q 30 -u 40 -l 250`. Обрезка по качеству (`cutadapt --quality-cutoff`, `fastp --cut_*`) не применяется. Результат записывается отдельно в `PRJNA1226555_fastp_min250`.


In [ ]:
from pathlib import Path
import gzip, json, os, shutil, subprocess, time
START=Path.cwd().resolve(); REPO=next((p for p in (START,*START.parents) if (p/'notebooks').is_dir()),None)
if REPO is None: raise RuntimeError(f'cannot find repo from {START}')
VOLUME=Path(os.environ.get('BCR_VOLUME','/data/user/epishkin'))
if not (VOLUME/'raw').is_dir(): VOLUME=REPO
ENV=Path(os.environ.get('BCR_ENV','/opt/conda/envs/bcr_env'))
if not ENV.is_dir(): ENV=Path('/Users/epishkin/mamba/envs/bcr_env')
RUN='SRR32426580'; RAW=VOLUME/'raw'/'PRJNA1226555'
ROOT=VOLUME/'results'/'PRJNA1226555'/'branches'/'fastp_q30_u40_min250'
R1=RAW/f'{RUN}_1.fastq.gz'; R2=RAW/f'{RUN}_2.fastq.gz'
def tool(name):
 p=ENV/'bin'/name
 if p.is_file(): return p
 q=shutil.which(name)
 if q:return Path(q)
 raise FileNotFoundError(name)
def fqcount(path):
 with gzip.open(path,'rt') as h:n=sum(1 for _ in h)
 assert n%4==0,path
 return n//4
def run(cmd,out,err,outputs=(),heartbeat=30):
 out=Path(out);err=Path(err);out.parent.mkdir(parents=True,exist_ok=True);started=time.monotonic()
 with out.open('w') as oh,err.open('w') as eh:
  proc=subprocess.Popen([str(x) for x in cmd],stdout=oh,stderr=eh,text=True);print(f'PID={proc.pid}',flush=True)
  while proc.poll() is None:
   sizes=' '.join(f'{Path(x).name}={Path(x).stat().st_size/1e6:.1f}MB' for x in outputs if Path(x).exists())
   print(f'PID={proc.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}',flush=True);time.sleep(heartbeat)
 if proc.returncode:raise RuntimeError(f'rc={proc.returncode}; see {err}')
for x in (R1,R2):assert x.is_file(),x
print('ROOT',ROOT)


In [ ]:
QUALITY_PHRED=30; UNQUALIFIED_PERCENT_LIMIT=40; MIN_LENGTH=250; ADAPTER_TIMES=2; ADAPTER_MIN_OVERLAP=10
ILLUMINA_ADAPTER_R1='AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT'; ILLUMINA_ADAPTER_R2='GATCGGAAGAGCACACGTCTGAACTCCAGTCAC'
BASE=ROOT/'trimmed';FQ=BASE/'fastq';LOG=BASE/'logs';REPORTS=BASE/'fastp_reports';TMP=BASE/'adapter_only_tmp'
shutil.rmtree(BASE,ignore_errors=True)
for d in (FQ,LOG,REPORTS,TMP):d.mkdir(parents=True)
a1=TMP/f'{RUN}_1.adapter.fastq.gz';a2=TMP/f'{RUN}_2.adapter.fastq.gz';t1=FQ/f'{RUN}_1.trim.fastq.gz';t2=FQ/f'{RUN}_2.trim.fastq.gz'
run([tool('cutadapt'),'--times',str(ADAPTER_TIMES),'-O',str(ADAPTER_MIN_OVERLAP),'--compression-level','1','-a',ILLUMINA_ADAPTER_R1,'-A',ILLUMINA_ADAPTER_R2,'--json',LOG/f'{RUN}.cutadapt.json','-o',a1,'-p',a2,R1,R2],LOG/f'{RUN}.cutadapt.stdout.log',LOG/f'{RUN}.cutadapt.stderr.log',[a1,a2])
run([tool('fastp'),'-i',a1,'-I',a2,'-o',t1,'-O',t2,'-q',str(QUALITY_PHRED),'-u',str(UNQUALIFIED_PERCENT_LIMIT),'-l',str(MIN_LENGTH),'--disable_adapter_trimming','--disable_trim_poly_g','-w','4','-j',REPORTS/f'{RUN}.fastp.json','-h',REPORTS/f'{RUN}.fastp.html'],LOG/f'{RUN}.fastp.stdout.log',LOG/f'{RUN}.fastp.stderr.log',[t1,t2])
a1.unlink();a2.unlink();assert fqcount(t1)==fqcount(t2)
d=json.loads((REPORTS/f'{RUN}.fastp.json').read_text());fr=d['filtering_result'];summary={'semantics':'cutadapt adapters-only; fastp whole-read filtering; no quality trimming','raw_pairs':fqcount(R1),'trimmed_pairs':fqcount(t1),'retention':fqcount(t1)/fqcount(R1),'qualified_quality_phred':QUALITY_PHRED,'unqualified_percent_limit':UNQUALIFIED_PERCENT_LIMIT,'minimum_length':MIN_LENGTH,'low_quality_reads':fr['low_quality_reads'],'too_short_reads':fr['too_short_reads']}
(BASE/'trim_summary.json').write_text(json.dumps(summary,indent=2)+'\n');print(summary)
